# Earth Departure

Goals:
- Departure Geometry (rla, dla)
- Escape orbit and burn
- Parking orbit and location
- Launch site and opportunities
- Propagation

In [ ]:

import numpy as np
from cesium.constants import M_TO_KM, RAD_TO_DEG
from cesium.math.time import jd_to_greg
from cesium.orbital.interplanetary import departure_characteristics
from cesium.orbital.transformations import ECI_from_HCI
from cesium.plotting import plot_3d
from cesium.world.constants import R_EARTH
from numpy.lib.npyio import NpzFile
from numpy.linalg import norm


## Loading trajectory
From `interplanetary.ipynb`

In [ ]:

MISSION_NAME = "birthday-christmas"
mission: NpzFile = np.load(f"trajectories/{MISSION_NAME}.mission", allow_pickle=False)

def print_mission(mission: NpzFile):
    print(f"Mission: {MISSION_NAME}")
    print("Heliocentric reference times")
    for label, key in (("Lambert departure (Earth reference)", "lambert_departure_jd"), ("Lambert arrival (Mars reference)", "lambert_arrival_jd")):
        print(f"{label}: " + "{:04.0f}-{:02.0f}-{:02.0f} {:02.0f}:{:02.0f}:{:06.3f}".format(*jd_to_greg(float(mission[key]))))
    print(f"Flight duration: {float(mission['tof_s']) / 86400:.4f} days")
    print(f"Samples: {mission['time_s'].size} (seconds since {mission['time_origin'].item()})")
    # print(f"Timesteps ranging from {mission['dt_s'].min():.3f}-{mission['dt_s'].max():.3f} s ")
    print(f"Frame: {mission['frame'].item()}; positions: {mission['position_units'].item()}; velocities: {mission['velocity_units'].item()}")
    print("\nStored fields:")
    for key in mission.files:
        info = mission[key].dtype

        if mission[key].dtype != float:
            info = '"' + mission[key] + '"'
        elif mission[key].shape == ():
            info = mission[key]
            
        # info = mission[key].dtype if mission[key].dtype == float else mission[key]
        print(f"  {key:24s} {mission[key].shape!s:14s} {info}")
print_mission(mission)

v_inf_hci = (mission["satellite_rv_hci_km"][0, 3:] - mission["earth_rv_hci_km"][0, 3:]) / M_TO_KM
v_inf_eci = ECI_from_HCI() @ v_inf_hci
c3, rla, dla = departure_characteristics(v_inf_eci * M_TO_KM)
print(f"\nEarth departure target: v_inf = {norm(v_inf_eci) * M_TO_KM:.4f} km/s; C3 = {c3:.4f} km²/s²")
print(f"RLA = {rla * RAD_TO_DEG:.4f} deg; DLA = {dla * RAD_TO_DEG:.4f} deg")


Mission: birthday-christmas
Heliocentric reference times
Lambert departure (Earth reference): 2035-06-13 23:25:00.000
Lambert arrival (Mars reference): 2035-12-25 00:00:00.000
Flight duration: 194.0243 days
Samples: 1942 (seconds since lambert_departure)
Frame: HCI; positions: km; velocities: km/s

Stored fields:
  lambert_departure_jd     ()             2464492.4756944445
  lambert_arrival_jd       ()             2464686.5
  tof_s                    ()             16763699.99999553
  time_s                   (1942,)        float64
  dt_s                     (1941,)        float64
  earth_rv_hci_km          (1942, 6)      float64
  mars_rv_hci_km           (1942, 6)      float64
  satellite_rv_hci_km      (1942, 6)      float64
  frame                    ()             "HCI"
  position_units           ()             "km"
  velocity_units           ()             "km/s"
  time_origin              ()             "lambert_departure"
  trajectory_model         ()             "heliocentric_

In [ ]:
# Translate by Earth's position at each sample, then rotate HCI into ECI.
r_sat_eci_km = (mission["satellite_rv_hci_km"][:,0:3] - mission["earth_rv_hci_km"][:,0:3]) @ ECI_from_HCI().T
fig_earth = plot_3d(
    r_sat_eci_km, mission["time_s"],
    labels="Satellite", trace_colors="purple",
    bodies={"name": "Earth", "radius": R_EARTH*M_TO_KM, "color": "royalblue"},
    frames={"name": "ECI", "location": [0, 0, 0],
            "dcm_body_wrt_hci": np.eye(3), "length": 10000},
    title="Earth departure — 50,000 km view", units="km",
    azim=80, elev=30, show_end=False, show=False,
)
fig_earth.update_scenes(
    xaxis_range=[-50000, 50000], # km
    yaxis_range=[-50000, 50000], # km
    zaxis_range=[-50000, 50000], # km
)
fig_earth.show()